# 00_data_check
Goal: check if the REES46 data can answer my questions before building silver.

## General findings
- **Size:** 233,460,662 rows · 10 columns · 4 full months (2019-10-01 to 2020-01-31 UTC)
- **Grain:** one action (view / cart / purchase) by one user, on one product, at one moment, in one session
- **Key:** event_time + event_type + product_id + user_id + user_session · 13 rows have the same key but different values (decision in silver step)
- **Duplicates:** 380,454 exact copies (0.16%) → remove in silver


## Findings per column
- **event_time**
  - Nulls: 0
  - Distinct: 10,466,683
  - Values: 2019-10-01 to 2020-01-31, text ending in " UTC"
  - 4 full months: October 2019 to January 2020

- **event_type**
  - Nulls: 0
  - Distinct: 3
  - Values: view 219.8M (94.2%) · cart 10.0M (4.3%) · purchase 3.7M (1.6%)
  - Notes: no case/space problems

- **product_id**
  - Nulls: 0
  - Distinct: 273,355

- **category_id**
  - Nulls: 0
  - Distinct: 1,253
  - Notes: only 135 category names for 1,253 ids → check later

- **category_code**
  - Nulls: 47,547,518 (20.4%)
  - Distinct: 135
  - Decision: keep rows, fill with `unknown`

- **brand**
  - Nulls: 29,979,794 (12.8%), more than any single brand
  - Distinct: 5,324
  - Values: top 3 samsung, apple, xiaomi (electronics-heavy)
  - Notes: no case/space problems

- **price**
  - Nulls: 0 (all values convert to numbers, checked with try_cast)
  - Distinct: 120,130
  - Values: min 0, max 2,574.07 · price = 0 on 495,957 rows (views + carts, no purchases)
  - Decision: price 0 → NULL · type decimal(10,2) in silver

- **user_id**
  - Nulls: 0
  - Distinct: 10,002,385

- **user_session**
  - Nulls: 52
  - Distinct: 52,321,177
  - Decision: see Silver specs below

- **source_file**
  - Added by me: the month file each row came from

- **Not in the data:** order_id, quantity, device, country

- **Monthly values:**
  - October:
    - users: 3,022,290 · sessions: 9,244,421
  - November:
    - users: 3,696,117 · sessions: 13,776,050
  - December:
    - users: 4,577,232 · sessions: 15,581,360
  - January:
    - users: 4,385,985 · sessions: 13,847,854
  - October is much smaller than the other months (I will check this later)
  - December is the peak (expected, holiday season)

## Silver specs

```
price:               text -> decimal(10,2)   0 -> null
category_code  :     null -> 'unknown'
event_time     :     text -> timestamp
full duplicates:     380,454 exact copies -> delete 
user_session   :     null -> stays NULL
brand          :     null -> 'unknown'
key columns    :     13 rows have the same 5 key columns but different values in other columns -> check in silver step
```

**Next step:**

- open 01_silver · check the 13 rows · write silver · expect 233,080,208 rows


## 1. Raw files
Download the four monthly CSV files into a Unity Catalog volume.

In [0]:
%sql

-- IF NOT EXISTS: safe to run again
create volume if not exists 
dev.ecommerce.raw_files

In [0]:
import os                  # tools for folders and files
import urllib.request      # tool to download files from the internet

# folder in the volume for this data source
folder = "/Volumes/dev/ecommerce/raw_files/rees46"

# source file: October 2019
url = "https://data.rees46.com/datasets/marketplace/2019-Oct.csv.gz"

# full path of the saved file = folder + file name
target = folder + "/2019-Oct.csv.gz"

# create folder; no error if it exists (safe to rerun)
os.makedirs(folder, exist_ok = True)  

# download the file from the URL into the volume
urllib.request.urlretrieve(url, target)



In [0]:
# check: file size in GB, should not be 0
os.path.getsize(target) / 1e9 

Test first with one month (October) to check the format.

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", True)
    .load("/Volumes/dev/ecommerce/raw_files/rees46/2019-Oct.csv.gz")
)

df.display()


In [0]:
df.createOrReplaceTempView("oct_raw")

Download the other three months.

In [0]:
import urllib.request  # tool to download files from the internet

# folder in the volume for this data source
folder = "/Volumes/dev/ecommerce/raw_files/rees46"

# all files have the same address, only the month changes
base_url = "https://data.rees46.com/datasets/marketplace/"

# the 3 months still missing (October is already there)
months = ["2019-Nov", "2019-Dec", "2020-Jan"]


# for each month: build the address and download the file

for month in months:
    url = base_url + month + ".csv.gz"
    target = folder + "/" + month + ".csv.gz"
    urllib.request.urlretrieve(url, target)

In [0]:
import os  # tools for folders and files

# check: list every file in the folder with its size in GB
for file_name in sorted(os.listdir(folder)):
    size_gb = os.path.getsize(folder + "/" + file_name) / 1e9
    print(file_name, round(size_gb, 2))

## 2. Bronze table
Read all four files as they are (all columns as text) and add `source_file`, the month file each row came from.

In [0]:
from pyspark.sql.functions import col 

df = (
    spark.read
    .format("csv")
    .option("header",True)
    .load("/Volumes/dev/ecommerce/raw_files/rees46/")
    .select("*", col("_metadata.file_name").alias("source_file"))
)

df.display()

In [0]:
# medallion architecture: save the raw data as it is, in a bronze table

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.ecommerce.events_bronze")

## 3. Checks
### Grain and key

In [0]:
%sql
select * from events_bronze

In [0]:
%sql
 select count(*), count(DISTINCT event_time, event_type, product_id, user_id, user_session) as key_col from events_bronze

Rows with a NULL in any key column.

In [0]:
%sql
-- rows where at least one key column is NULL
SELECT count(*)
FROM dev.ecommerce.events_bronze
WHERE event_time IS NULL
   OR event_type IS NULL
   OR product_id IS NULL
   OR user_id IS NULL
   OR user_session IS NULL

### Duplicates

In [0]:
%sql

-- Check for exact duplicates: group the rows by all 9 data columns, so identical rows fall into the same group,
-- and count how many times each row appears (rn). Keep only groups with rn > 1. Each group has 1 original
-- and rn - 1 extra copies, so sum(rn - 1) is the number of duplicate rows to remove in silver.
-- Result: 380,454 exact duplicates (about 0.16% of 233,460,662 rows).


with x as (
select count(*) as rn  ,  event_time, event_type, product_id,category_id, category_code,brand,price, user_id, user_session  from events_bronze
group by event_time, event_type, product_id,category_id, category_code,brand,price, user_id, user_session
having rn>1
)
select sum(rn-1) from x 


### Distinct values per column

In [0]:
%sql
select count(distinct user_id), approx_count_distinct(user_id)
from events_bronze

In [0]:
%sql
select 
count(distinct event_time)    as distinct_event_time,
count(distinct event_type)    as distinct_event_type,
count(distinct product_id)    as distinct_product_id,
count(distinct category_id)   as distinct_category_id,
count(distinct category_code) as distinct_category_code,
count(distinct brand)         as distinct_brand,
count(distinct price)         as distinct_price,
count(distinct user_id)       as distinct_user_id,
count(distinct user_session)  as distinct_user_session
from events_bronze





### Values: event types, date range, brands, prices

In [0]:
%sql
select count(*), event_type from events_bronze
group by event_type;

select min(event_time) as min,
max(event_time) as max from events_bronze;

select count(*), brand from events_bronze
group by brand
order by count(*) desc;


In [0]:
%sql
select
count(distinct brand) , count(distinct lower(trim(brand))), 
count(distinct event_type) , count(distinct lower(trim(event_type)))

from events_bronze;


select 
count(*) from events_bronze
where try_cast(price as double) is null;

select 
min(try_cast(price as double)) as min,
max(try_cast(price as double)) as max from events_bronze ;

select count(*) , event_type  from events_bronze
where try_cast(price as double) = 0
group by event_type

In [0]:
%sql
select count(*), event_type from events_bronze 
group by event_type

In [0]:
%sql
SELECT 
max(event_time) as max_event_day,
min(event_time) as min_event_day from events_bronze

### Users and sessions per month

In [0]:
%sql
select 
date_format(event_time, 'yyyy-MM') as month,
count(distinct user_id) as nb_of_users,
count(distinct user_session) as nb_of_session from events_bronze 
group by month 
order by month

### Table info

In [0]:
%sql
DESCRIBE EXTENDED dev.ecommerce.events_bronze

In [0]:
%sql
SELECT source_file, COUNT(*) FROM events_bronze GROUP BY source_file

In [0]:
%sql

SHOW TABLES IN dev.ecommerce